# 04 · Churn Prediction

**Goal:** Build a binary classifier that predicts whether a customer
will stop buying.

**Definition:** A customer is *churned* if they have not purchased
in the last **90 days** of the dataset window.

Models compared:
- Logistic Regression (baseline, interpretable)
- Random Forest (handles non-linearity)
- XGBoost (optional — install `xgboost`)

## 0 · Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve,
)

from src.churn_model import ChurnModel
from src.features    import get_feature_sets

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "axes.titleweight": "bold"})
PALETTE = ["#4361EE", "#3A0CA3", "#7209B7", "#F72585", "#4CC9F0"]
sns.set_theme(style="whitegrid")


## 1 · Load RFM features

In [ ]:
rfm = pd.read_csv("../data/processed/rfm_features.csv")
print(f"Shape: {rfm.shape}")
print(f"Churn rate: {rfm['Churned'].mean()*100:.1f}%")
rfm[["Recency","Frequency","Monetary","Churned"]].describe().round(2)


## 2 · Class imbalance check

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = rfm["Churned"].value_counts()
bars = ax.bar(["Active", "Churned"], counts.values,
              color=[PALETTE[0], "#e63946"])
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f"{val:,}\n({val/len(rfm)*100:.1f}%)", ha="center")
ax.set_title("Class Distribution")
ax.set_ylabel("# Customers")
plt.tight_layout()
plt.savefig("../reports/figures/churn_class_balance.png", bbox_inches="tight")
plt.show()


## 3 · Train models

In [ ]:
feat_cols = get_feature_sets()["churn"]
print("Features:", feat_cols)

churn_model = ChurnModel(test_size=0.2, random_state=42)
churn_model.fit(rfm, feature_cols=feat_cols)


## 4 · Model comparison table

In [ ]:
eval_df = churn_model.evaluation_report()
print(eval_df.to_string())

fig, ax = plt.subplots(figsize=(9, 4))
eval_df[["Accuracy","Precision","Recall","F1","ROC-AUC"]].plot(
    kind="bar", ax=ax, color=PALETTE[:5], edgecolor="white")
ax.set_ylim(0, 1)
ax.set_title("Model Comparison — All Metrics")
ax.tick_params(axis="x", rotation=0)
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("../reports/figures/churn_model_comparison.png", bbox_inches="tight")
plt.show()


## 5 · Confusion matrix

In [ ]:
cm = churn_model.confusion()
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Active", "Churned"],
            yticklabels=["Active", "Churned"])
ax.set_title(f"Confusion Matrix — {churn_model.best_model_name}")
ax.set_ylabel("Actual"); ax.set_xlabel("Predicted")
plt.tight_layout()
plt.savefig("../reports/figures/churn_confusion_matrix.png", bbox_inches="tight")
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Positives  (correctly caught churners): {tp}")
print(f"False Negatives (missed churners):           {fn}")
print(f"False Positives (unnecessary campaigns):     {fp}")


## 6 · ROC curve

In [ ]:
feat_cols = churn_model.feature_cols
X = rfm[feat_cols]
y = rfm["Churned"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                           stratify=y, random_state=42)

fig, ax = plt.subplots(figsize=(7, 6))
for name, metrics in churn_model.results.items():
    model = metrics["model"]
    proba = model.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, proba)
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {roc_auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_title("ROC Curves — Churn Prediction")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("../reports/figures/churn_roc.png", bbox_inches="tight")
plt.show()


## 7 · Precision-Recall curve

In [ ]:
best_proba = churn_model.best_model.predict_proba(X_te)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_te, best_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(recall, precision, color=PALETTE[0], lw=2)
axes[0].set_title(f"Precision-Recall — {churn_model.best_model_name}")
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision")

f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-9)
best_idx  = f1_scores.argmax()
axes[1].plot(thresholds, precision[:-1], label="Precision", color=PALETTE[0])
axes[1].plot(thresholds, recall[:-1],    label="Recall",    color=PALETTE[3])
axes[1].plot(thresholds, f1_scores,      label="F1",        color=PALETTE[2], ls="--")
axes[1].axvline(thresholds[best_idx], color="green", ls=":", lw=1.5,
                label=f"Best threshold = {thresholds[best_idx]:.2f}")
axes[1].set_title("Metrics vs Threshold")
axes[1].set_xlabel("Decision Threshold")
axes[1].legend()

plt.tight_layout()
plt.savefig("../reports/figures/churn_pr_curve.png", bbox_inches="tight")
plt.show()
print(f"Best F1 threshold: {thresholds[best_idx]:.3f}")


## 8 · Feature importances

In [ ]:
fi = churn_model.feature_importances()
if fi is not None:
    fig, ax = plt.subplots(figsize=(8, 4))
    fi.sort_values().plot(kind="barh", ax=ax, color=PALETTE[4])
    ax.set_title(f"Feature Importances — {churn_model.best_model_name}")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.savefig("../reports/figures/churn_feature_importance.png", bbox_inches="tight")
    plt.show()
    print(fi.to_string())
else:
    print("Feature importances not available for this model type.")


## 9 · Add churn probabilities to RFM table

In [ ]:
rfm["ChurnProbability"] = churn_model.predict_proba(rfm)
rfm["ChurnRiskBand"]    = pd.cut(
    rfm["ChurnProbability"],
    bins=[0, 0.4, 0.7, 1.0],
    labels=["Low", "Medium", "High"],
)

print(rfm["ChurnRiskBand"].value_counts())
rfm[["CustomerID","Recency","Frequency","Monetary","ChurnProbability","ChurnRiskBand"]].head(10)


## 10 · Save model

In [ ]:
churn_model.save("../models/churn_rf_model.pkl")
rfm.to_csv("../data/processed/customer_segments.csv", index=False)
print("Model and enriched segment file saved.")


## Summary

- **Random Forest** edges out Logistic Regression on F1 (0.53 vs 0.50).
- **ROC-AUC ~ 0.80** — the model is a solid discriminator but not perfect.
- **Frequency** is the most important feature — customers who buy rarely are
  most likely to churn.
- A threshold of ~0.45 (vs default 0.5) can recover recall at the cost of
  some precision — useful if the campaign cost is low.
- Consider **SMOTE** or `class_weight='balanced'` to improve recall further.